In [ ]:
# revT2V -- Colab inference server
# Boots the method registry on Colab's GPU, starts server/app.py with
# uvicorn, and opens a public URL via cloudflared. Run every cell in order
# (Runtime > Run all). See server/API.md for what the frontend talks to.

# 1. GPU check -- every method here needs one
import torch

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > GPU, then Run all again."
print("GPU:", torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")

In [ ]:
# 2. Clone/pull the repo and install it
# No %load_ext autoreload here on purpose -- Colab's Python 3.13 breaks it.
REPO_URL = "https://github.com/silentlooop/revT2V.git"
REPO_DIR = "/content/revT2V"

!git clone -q {REPO_URL} {REPO_DIR} 2>/dev/null || git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!git log --oneline -1
!pip install -q -e .
!pip install -q fastapi uvicorn nest_asyncio imageio-ffmpeg

In [ ]:
# 3. HF token + API key
# HF_TOKEN comes from a Colab secret (left sidebar, key icon) -- add one
# named HF_TOKEN with read access to silentlooop/revt2v-ckpt and turn on
# notebook access for it. API_KEY is optional: set it yourself below, or
# leave it blank to get a random one (printed in step 6) that the frontend
# must send back as X-API-Key.
import os
import secrets

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
assert os.environ["HF_TOKEN"], "Add an HF_TOKEN secret (left sidebar) and enable notebook access"

API_KEY = os.environ.get("API_KEY") or secrets.token_urlsafe(16)
os.environ["API_KEY"] = API_KEY
print("API key (also printed again with the URL in step 6):", API_KEY)

In [ ]:
# 4. Load the teacher once, and every available trained method
# get_bank() loads the teacher pipeline and tries each trained method's
# checkpoint; a missing checkpoint is reported, not an error (attn_injection
# just runs zero-shot, and the matching entry in GET /methods comes back
# available: false for conv_lora if its checkpoint is missing).
from server.app import get_bank

bank = get_bank()
print("Methods ready:", bank.methods)
for method in ("conv_lora", "attn_injection"):
    status = "available" if method in bank.adapters else ("zero-shot" if method == "attn_injection" else "NOT FOUND (checkpoint missing)")
    print(f"  {method}: {status}")

In [ ]:
# 5. Start the server in a background thread
import threading
import time

import nest_asyncio
import requests
import uvicorn

nest_asyncio.apply()

config = uvicorn.Config("server.app:app", host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
threading.Thread(target=server.run, daemon=True).start()

for _ in range(60):
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=2).status_code == 200:
            print("Server is up.")
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Server did not come up within 60s -- check the log output above")

In [ ]:
# 6. Open a public tunnel with cloudflared
# cloudflared's free quick tunnels need no account or config -- just a
# binary and this one command. The URL changes every time this cell (or the
# whole notebook) restarts; re-run the test cell below if it does.
import re
import subprocess
import time

import requests

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

SERVER_URL = None
deadline = time.time() + 30
while time.time() < deadline:
    line = tunnel.stdout.readline()
    if not line:
        continue
    match = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
    if match:
        SERVER_URL = match.group(0)
        break

assert SERVER_URL, "Could not find the tunnel URL in cloudflared's output within 30s"

health = requests.get(f"{SERVER_URL}/health", timeout=10)
assert health.status_code == 200, f"Tunnel is up but /health through it failed: {health.status_code} {health.text}"

print("=" * 50)
print("SERVER URL: ", SERVER_URL)
print("API KEY:    ", API_KEY)
print("Paste both into the frontend settings.")
print("=" * 50)

In [ ]:
# 7. Test cell: one request through the public URL
# Submits a job exactly the way the frontend will, polls it, and plays the
# result inline. compare_with_target=True also exercises the metrics code path.
# Methods must be ids from GET /methods (attn_injection, conv_oracle, conv_lora
# -- "teacher" is only ever an internal comparison reference, not selectable).
import time

from IPython.display import Video, display

headers = {"X-API-Key": API_KEY}

resp = requests.post(
    f"{SERVER_URL}/generate",
    headers=headers,
    json={
        "prompt": "a red ball bouncing on a wooden floor",
        "methods": ["attn_injection", "conv_oracle"],
        "steps": 25,
        "compare_with_target": True,
    },
)
resp.raise_for_status()
job_id = resp.json()["job_id"]
print("job_id:", job_id, "| position:", resp.json()["position"])

while True:
    job = requests.get(f"{SERVER_URL}/jobs/{job_id}", headers=headers).json()
    print(f"status={job['status']} progress={job['progress']}")
    if job["status"] in ("done", "error"):
        break
    time.sleep(1.5)

assert job["status"] == "done", job.get("error")
for result in job["results"]:
    print(result["method"], result["metrics"])
    display(Video(f"{SERVER_URL}{result['video_url']}", embed=False))

In [ ]:
# 8. Keep alive / logs
# Leave this cell running while you use the frontend -- it just reports the
# queue so you can tell the server is still alive. The tunnel URL changes
# every time this notebook's runtime restarts; if the frontend stops
# connecting, re-run from step 5 (or step 2 if the runtime was fully reset)
# and paste the new URL.
import time

try:
    while True:
        health = requests.get("http://127.0.0.1:8000/health", timeout=5).json()
        print(time.strftime("%H:%M:%S"), health)
        time.sleep(10)
except KeyboardInterrupt:
    print("Stopped. The server thread and tunnel are still running in the background.")